
# LoRA catastrophic-forgetting training dynamics

This notebook loads the output directories produced by `train.py` and compares the four protocols:

- `joint`
- `W_then_w`
- `w_then_W`
- `matched_two_stage`

It is designed for the metrics recorded during true single-sample SGD. If several seeds are available for the same configuration, the plots show the mean trajectory and a ±1 standard-deviation band. With one seed, the raw trajectory is shown.

The main plots are:

1. old-task error \(E_S\),
2. new-task error \(E_{S'}\),
3. old-task error with LoRA disabled \(E_S^{W\text{-only}}\),
4. forgetting \(F_S\),
5. overlaps \(\rho_W,\rho_w\),
6. reconstruction errors \(\epsilon_W,\epsilon_w\),
7. effective noise \(\Delta_{\rm eff}\),
8. cross-contamination observables,
9. gradient norms and gradient interference,
10. SGD training losses.


In [1]:

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# USER CONFIGURATION
# ------------------------------------------------------------------

# Change this if the notebook is not launched from the project root.
RESULTS_ROOT = Path("../results/training_dynamics")

# Optional output directory for figures.
FIGURE_DIR = Path("../figures/training_dynamics")
SAVE_FIGURES = False

# Select the scientific configuration to compare.
SELECT = {
    "d": 200,
    "T": 5,
    "kappa_star": 1.0,
    "kappa": 1.0,
    # Set alpha / alpha_prime to None to inspect all values.
    "alpha": 2.0,
    "alpha_prime": 2.0,
    "activation": "softmax",
}

PROTOCOL_ORDER = [
    "joint",
    "W_then_w",
    "w_then_W",
    "matched_two_stage",
]

PROTOCOL_LABELS = {
    "joint": "Joint",
    "W_then_w": r"$W \rightarrow w$",
    "w_then_W": r"$w \rightarrow W$",
    "matched_two_stage": "Matched two-stage",
}

# x-axis choices:
#   "step"          -> raw number of SGD samples/updates
#   "step_over_d"   -> t / d
#   "step_over_d2"  -> t / d^2
X_AXIS = "step"

# "linear", "log", or "symlog".
# symlog keeps the t=0 checkpoint and resolves both O(d) and O(d^2).
X_SCALE = "symlog"

# Draw generic reference scales t=d and t=d^2 when possible.
SHOW_D_SCALES = True

FIGURE_DIR.mkdir(parents=True, exist_ok=True)


ModuleNotFoundError: No module named 'pandas'

## 1. Load all runs

In [ ]:

def _read_config(run_dir: Path):
    path = run_dir / "config.json"
    if not path.exists():
        return None
    with path.open("r") as f:
        return json.load(f)


def _read_metrics(run_dir: Path):
    csv_path = run_dir / "metrics.csv"
    if csv_path.exists():
        return pd.read_csv(csv_path)

    npz_path = run_dir / "metrics.npz"
    if npz_path.exists():
        data = np.load(npz_path, allow_pickle=False)
        return pd.DataFrame({k: data[k] for k in data.files})

    return None


def discover_runs(results_root: Path) -> pd.DataFrame:
    """
    Discover every run containing config.json and metrics.csv/metrics.npz.
    Partial runs are included if metric files exist.
    """
    rows = []

    if not results_root.exists():
        raise FileNotFoundError(
            f"RESULTS_ROOT does not exist: {results_root.resolve()}"
        )

    for config_path in sorted(results_root.rglob("config.json")):
        run_dir = config_path.parent
        config = _read_config(run_dir)
        metrics = _read_metrics(run_dir)

        if config is None or metrics is None or len(metrics) == 0:
            continue

        model = config.get("model", {})
        protocol = config.get("protocol", {})
        training = config.get("training", {})
        seeds = config.get("seeds", {})

        rows.append({
            "run_dir": run_dir,
            "experiment": protocol.get("experiment"),
            "d": model.get("d"),
            "T": model.get("T"),
            "kappa_star": model.get("kappa_star"),
            "kappa": model.get("kappa"),
            "alpha": protocol.get("alpha"),
            "alpha_prime": protocol.get("alpha_prime"),
            "p_finetune": protocol.get("p_finetune"),
            "activation": model.get("activation"),
            "seed": seeds.get("main_seed"),
            "schedule_seed": seeds.get("schedule_seed"),
            "lr_W": training.get("lr_W"),
            "lr_w": training.get("lr_w"),
            "log_every": training.get("log_every"),
            "n_checkpoints": len(metrics),
            "last_step": int(metrics["step"].iloc[-1]),
        })

    return pd.DataFrame(rows)


runs = discover_runs(RESULTS_ROOT)
print(f"Discovered {len(runs)} run(s) under {RESULTS_ROOT.resolve()}")
runs


## 2. Select one parameter setting

In [ ]:

def select_runs(runs: pd.DataFrame, selection: dict) -> pd.DataFrame:
    selected = runs.copy()

    for key, wanted in selection.items():
        if wanted is None:
            continue

        if key not in selected.columns:
            raise KeyError(f"Selection key {key!r} not present in run metadata.")

        if isinstance(wanted, float):
            selected = selected[
                np.isclose(selected[key].astype(float), wanted, equal_nan=False)
            ]
        else:
            selected = selected[selected[key] == wanted]

    selected = selected.copy()
    selected["experiment"] = pd.Categorical(
        selected["experiment"],
        categories=PROTOCOL_ORDER,
        ordered=True,
    )
    selected = selected.sort_values(["experiment", "seed"])
    return selected


selected_runs = select_runs(runs, SELECT)

if selected_runs.empty:
    raise RuntimeError(
        "No runs match SELECT. Inspect the `runs` table above and modify SELECT."
    )

display_cols = [
    "experiment", "seed", "d", "T", "kappa_star", "kappa",
    "alpha", "alpha_prime", "lr_W", "lr_w",
    "n_checkpoints", "last_step", "run_dir",
]
selected_runs[display_cols]


In [ ]:

seed_counts = (
    selected_runs.groupby("experiment", observed=True)["seed"]
    .nunique()
    .reindex(PROTOCOL_ORDER)
    .dropna()
)
seed_counts


## 3. Build one long dataframe of trajectories

In [ ]:

def load_selected_trajectories(selected_runs: pd.DataFrame) -> pd.DataFrame:
    frames = []

    for _, meta in selected_runs.iterrows():
        metrics = _read_metrics(Path(meta["run_dir"]))
        metrics = metrics.copy()

        for col in [
            "experiment", "seed", "d", "T", "kappa_star", "kappa",
            "alpha", "alpha_prime", "lr_W", "lr_w",
        ]:
            metrics[col] = meta[col]

        metrics["run_dir"] = str(meta["run_dir"])
        frames.append(metrics)

    out = pd.concat(frames, ignore_index=True)

    numeric_progress = [
        "step", "samples_seen", "step_over_d", "step_over_d2",
        "n_S_seen", "n_Sprime_seen", "n_S_over_d2", "n_Sprime_over_d",
    ]
    for col in numeric_progress:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


traj = load_selected_trajectories(selected_runs)
print(f"{len(traj):,} logged checkpoints loaded.")
print(
    f"{traj['experiment'].nunique()} protocol(s), "
    f"{traj['seed'].nunique()} unique seed value(s)."
)

sorted(traj.columns.tolist())



## 4. Plotting helpers

For several seeds, aggregation is performed at identical logged `step` values within each protocol. Extra phase-boundary checkpoints therefore remain visible.

The default `symlog` x-axis is useful because it preserves \(t=0\) while showing both the \(O(d)\) and \(O(d^2)\) time scales.


In [ ]:

def _x_label(x_col: str) -> str:
    return {
        "step": "SGD samples / updates",
        "step_over_d": r"$t/d$",
        "step_over_d2": r"$t/d^2$",
    }[x_col]


def aggregate_metric(df: pd.DataFrame, metric: str, x_col: str = X_AXIS):
    """Return mean/std/count at each x point for each protocol."""
    required = {"experiment", x_col, metric}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns required for {metric}: {sorted(missing)}")

    clean = df[["experiment", x_col, metric]].copy()
    clean[metric] = pd.to_numeric(clean[metric], errors="coerce")
    clean = clean.dropna(subset=[x_col, metric])

    return (
        clean.groupby(["experiment", x_col], observed=True)[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )


def _add_scale_markers(ax, df: pd.DataFrame, x_col: str):
    if not SHOW_D_SCALES or df.empty:
        return

    d_values = pd.to_numeric(df["d"], errors="coerce").dropna().unique()
    if len(d_values) != 1:
        return

    d = float(d_values[0])

    if x_col == "step":
        ax.axvline(d, linestyle=":", linewidth=1.0, alpha=0.6)
        ax.axvline(d * d, linestyle=":", linewidth=1.0, alpha=0.6)
    elif x_col == "step_over_d":
        ax.axvline(1.0, linestyle=":", linewidth=1.0, alpha=0.6)
        ax.axvline(d, linestyle=":", linewidth=1.0, alpha=0.6)
    elif x_col == "step_over_d2":
        ax.axvline(1.0 / d, linestyle=":", linewidth=1.0, alpha=0.6)
        ax.axvline(1.0, linestyle=":", linewidth=1.0, alpha=0.6)


def plot_metric(
    df: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    *,
    x_col: str = X_AXIS,
    x_scale: str = X_SCALE,
    yscale: str = "linear",
    ylim=None,
    zero_line: bool = False,
    save_name: str | None = None,
):
    agg = aggregate_metric(df, metric, x_col=x_col)

    fig, ax = plt.subplots(figsize=(8.5, 5.2))

    for experiment in PROTOCOL_ORDER:
        sub = agg[agg["experiment"] == experiment].sort_values(x_col)
        if sub.empty:
            continue

        line, = ax.plot(
            sub[x_col],
            sub["mean"],
            linewidth=2,
            label=PROTOCOL_LABELS.get(experiment, experiment),
        )

        band = sub["count"] >= 2
        if band.any():
            low = sub.loc[band, "mean"] - sub.loc[band, "std"].fillna(0.0)
            high = sub.loc[band, "mean"] + sub.loc[band, "std"].fillna(0.0)
            ax.fill_between(
                sub.loc[band, x_col],
                low,
                high,
                alpha=0.16,
                color=line.get_color(),
                linewidth=0,
            )

    if zero_line:
        ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.6)

    if x_scale == "symlog":
        ax.set_xscale("symlog", linthresh=1.0)
    else:
        ax.set_xscale(x_scale)

    ax.set_yscale(yscale)
    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.set_xlabel(_x_label(x_col))
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    _add_scale_markers(ax, df, x_col)
    fig.tight_layout()

    if SAVE_FIGURES and save_name:
        fig.savefig(FIGURE_DIR / save_name, dpi=200, bbox_inches="tight")

    plt.show()
    return fig, ax


## 5. Main performance / catastrophic-forgetting plots

In [ ]:
plot_metric(
    traj,
    "E_S",
    ylabel=r"$E_S$",
    title="Old-task test error — full current model",
    save_name="E_S.png",
)


In [ ]:
plot_metric(
    traj,
    "E_Sprime",
    ylabel=r"$E_{S'}$",
    title="Fine-tuning-task test error",
    save_name="E_Sprime.png",
)


In [ ]:
plot_metric(
    traj,
    "E_S_W_only",
    ylabel=r"$E_S^{W\mathrm{-only}}$",
    title="Old-task error with LoRA switched off",
    save_name="E_S_W_only.png",
)


In [ ]:
plot_metric(
    traj,
    "F_S",
    ylabel=r"$F_S$",
    title="Catastrophic forgetting relative to best previous old-task error",
    save_name="F_S.png",
)



Interpret \(E_S\) together with \(E_S^{W\text{-only}}\):

- if \(E_S\) rises while \(E_S^{W\text{-only}}\) remains low, forgetting is mainly **LoRA interference**;
- if both rise, the extensive-rank component \(W\) itself has drifted away from the old task.


## 6. Teacher recovery / representation learning

In [ ]:
plot_metric(
    traj,
    "rho_W",
    ylabel=r"$\rho_W$",
    title="Normalized extensive-rank overlap",
    ylim=(0, 1.05),
    save_name="rho_W.png",
)


In [ ]:
plot_metric(
    traj,
    "rho_w",
    ylabel=r"$\rho_w$",
    title="Normalized LoRA-vector overlap",
    ylim=(0, 1.05),
    save_name="rho_w.png",
)


In [ ]:
plot_metric(
    traj,
    "eps_W",
    ylabel=r"$\epsilon_W$",
    title="Extensive-rank reconstruction error",
    ylim=None,
    save_name="eps_W.png",
)


In [ ]:
plot_metric(
    traj,
    "eps_w",
    ylabel=r"$\epsilon_w$",
    title="Rank-one reconstruction error",
    ylim=None,
    save_name="eps_w.png",
)



## 7. Effective noise seen by LoRA

For the current noiseless teacher, \(\Delta=0\), so

\[
\Delta_{\rm eff}=Q_0+Q-2M=\epsilon_W.
\]

This is the nuisance inherited by the rank-one fine-tuning problem from imperfect learning of the extensive-rank component.


In [ ]:

plot_metric(
    traj,
    "Delta_eff",
    ylabel=r"$\Delta_{\rm eff}$",
    title="Effective noise inherited from the extensive-rank representation",
    save_name="Delta_eff.png",
)


## 8. Cross-contamination observables

In [ ]:
plot_metric(
    traj,
    "C_W_from_wstar",
    ylabel=r"$C_{W\leftarrow w_*}$",
    title="Does the extensive-rank block absorb the rank-one teacher direction?",
    zero_line=True,
    save_name="C_W_from_wstar.png",
)


In [ ]:
plot_metric(
    traj,
    "C_w_from_Wstar",
    ylabel=r"$C_{w\leftarrow W_*}$",
    title="Does the LoRA vector align with extensive-rank teacher structure?",
    zero_line=True,
    save_name="C_w_from_Wstar.png",
)


In [ ]:
plot_metric(
    traj,
    "C_W_w",
    ylabel=r"$C_{W,w}$",
    title="Student–student interaction between W and w",
    zero_line=True,
    save_name="C_W_w.png",
)


## 9. Gradient interference

In [ ]:
plot_metric(
    traj,
    "Gamma_W",
    ylabel=r"$\Gamma_W$",
    title="Task-gradient cosine in the extensive-rank block",
    ylim=(-1.05, 1.05),
    zero_line=True,
    save_name="Gamma_W.png",
)


In [ ]:
plot_metric(
    traj,
    "Gamma_w",
    ylabel=r"$\Gamma_w$",
    title="Task-gradient cosine in the LoRA block",
    ylim=(-1.05, 1.05),
    zero_line=True,
    save_name="Gamma_w.png",
)



\(\Gamma>0\) means the two task gradients are locally compatible, \(\Gamma<0\) means direct gradient conflict, and \(\Gamma\simeq0\) means approximately orthogonal directions.


## 10. Gradient magnitudes

In [ ]:
plot_metric(
    traj,
    "grad_W_S_norm",
    ylabel=r"$\|\nabla_W L_S\|_F$",
    title="Old-task gradient norm on W",
    yscale="log",
    save_name="grad_W_S_norm.png",
)


In [ ]:
plot_metric(
    traj,
    "grad_W_Sprime_norm",
    ylabel=r"$\|\nabla_W L_{S'}\|_F$",
    title="New-task gradient norm on W",
    yscale="log",
    save_name="grad_W_Sprime_norm.png",
)


In [ ]:
plot_metric(
    traj,
    "grad_w_S_norm",
    ylabel=r"$\|\nabla_w L_S\|_2$",
    title="Old-task gradient norm on w",
    yscale="log",
    save_name="grad_w_S_norm.png",
)


In [ ]:
plot_metric(
    traj,
    "grad_w_Sprime_norm",
    ylabel=r"$\|\nabla_w L_{S'}\|_2$",
    title="New-task gradient norm on w",
    yscale="log",
    save_name="grad_w_Sprime_norm.png",
)


## 11. SGD losses actually observed during training

In [ ]:
plot_metric(
    traj,
    "train_loss_mean_since_log",
    ylabel=r"mean sample loss",
    title="Mean SGD loss since previous checkpoint",
    save_name="train_loss_mean.png",
)


In [ ]:
plot_metric(
    traj,
    "train_loss_S_mean_since_log",
    ylabel=r"mean $L_S$",
    title="Mean old-task SGD loss since previous checkpoint",
    save_name="train_loss_S_mean.png",
)


In [ ]:
plot_metric(
    traj,
    "train_loss_Sprime_mean_since_log",
    ylabel=r"mean $L_{S'}$",
    title="Mean new-task SGD loss since previous checkpoint",
    save_name="train_loss_Sprime_mean.png",
)



## 12. Inspect one protocol/seed with its phase transition

The previous figures compare all four protocols. For debugging or interpreting a single curriculum, it is useful to show the actual phase boundary.


In [ ]:

def plot_single_run_with_phases(
    df: pd.DataFrame,
    *,
    experiment: str,
    seed: int,
    metric: str,
    ylabel: str,
    title: str,
    x_col: str = X_AXIS,
):
    exp_as_str = df["experiment"].astype(str)
    seed_num = pd.to_numeric(df["seed"], errors="coerce")

    sub = df[
        (exp_as_str == experiment) & (seed_num == seed)
    ].sort_values("step")

    if sub.empty:
        raise ValueError(
            f"No run found for experiment={experiment!r}, seed={seed}."
        )

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.plot(sub[x_col], sub[metric], linewidth=2)

    phase = sub["phase"].astype(str).to_numpy()
    change_idx = np.where(phase[1:] != phase[:-1])[0] + 1

    for idx in change_idx:
        x = sub.iloc[idx][x_col]
        ax.axvline(x, linestyle="--", linewidth=1.2, alpha=0.7)

    if X_SCALE == "symlog":
        ax.set_xscale("symlog", linthresh=1.0)
    else:
        ax.set_xscale(X_SCALE)

    ax.set_xlabel(_x_label(x_col))
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.25)
    _add_scale_markers(ax, sub, x_col)
    fig.tight_layout()
    plt.show()


# Example:
# plot_single_run_with_phases(
#     traj,
#     experiment="matched_two_stage",
#     seed=0,
#     metric="E_Sprime",
#     ylabel=r"$E_{S'}$",
#     title="Matched two-stage: fine-tuning error and phase transition",
# )


## 13. Compact final-value comparison

In [ ]:

def final_checkpoint_table(df: pd.DataFrame) -> pd.DataFrame:
    idx = df.groupby(["experiment", "seed"], observed=True)["step"].idxmax()
    final = df.loc[idx].copy()

    cols = [
        "experiment", "seed", "step",
        "E_S", "E_Sprime", "E_S_W_only", "F_S",
        "rho_W", "rho_w", "eps_W", "eps_w", "Delta_eff",
        "C_W_from_wstar", "C_w_from_Wstar", "C_W_w",
    ]
    cols = [c for c in cols if c in final.columns]
    return final[cols].sort_values(["experiment", "seed"])


finals = final_checkpoint_table(traj)
finals


In [ ]:

numeric_cols = [
    c for c in finals.columns
    if c not in {"experiment", "seed"}
]

final_summary = (
    finals.groupby("experiment", observed=True)[numeric_cols]
    .agg(["mean", "std", "count"])
)

final_summary



## 14. Optional: compare different \(\alpha,\alpha'\) values

Set `SELECT["alpha"] = None` and/or `SELECT["alpha_prime"] = None` near the top, rerun the loading cells, and use this helper to inspect final performance as a function of sample budget.


In [ ]:

def final_metric_vs_budget(
    all_runs: pd.DataFrame,
    *,
    metric: str,
    vary: str = "alpha",
    fixed_selection: dict | None = None,
):
    if vary not in {"alpha", "alpha_prime"}:
        raise ValueError("vary must be 'alpha' or 'alpha_prime'.")

    fixed_selection = dict(fixed_selection or {})
    fixed_selection[vary] = None
    chosen = select_runs(all_runs, fixed_selection)

    rows = []
    for _, meta in chosen.iterrows():
        metrics = _read_metrics(Path(meta["run_dir"]))
        if metric not in metrics.columns:
            continue
        last = metrics.iloc[-1]
        rows.append({
            "experiment": str(meta["experiment"]),
            "seed": meta["seed"],
            vary: meta[vary],
            metric: float(last[metric]),
        })

    data = pd.DataFrame(rows)
    if data.empty:
        raise RuntimeError("No matching final metrics found.")

    agg = (
        data.groupby(["experiment", vary], observed=True)[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(8.5, 5.2))

    for experiment in PROTOCOL_ORDER:
        sub = agg[agg["experiment"] == experiment].sort_values(vary)
        if sub.empty:
            continue

        line, = ax.plot(
            sub[vary],
            sub["mean"],
            marker="o",
            linewidth=2,
            label=PROTOCOL_LABELS.get(experiment, experiment),
        )

        band = sub["count"] >= 2
        if band.any():
            ax.fill_between(
                sub.loc[band, vary],
                sub.loc[band, "mean"] - sub.loc[band, "std"].fillna(0.0),
                sub.loc[band, "mean"] + sub.loc[band, "std"].fillna(0.0),
                color=line.get_color(),
                alpha=0.16,
                linewidth=0,
            )

    ax.set_xlabel(r"$\alpha$" if vary == "alpha" else r"$\alpha'$")
    ax.set_ylabel(metric)
    ax.set_title(f"Final {metric} versus {vary}")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()


# Example:
# final_metric_vs_budget(
#     runs,
#     metric="E_Sprime",
#     vary="alpha_prime",
#     fixed_selection={
#         "d": 200,
#         "T": 5,
#         "kappa_star": 1.0,
#         "kappa": 1.0,
#         "alpha": 2.0,
#         "activation": "softmax",
#     },
# )



## Suggested first inspection order

For a first run, inspect:

1. \(E_S\) and \(E_{S'}\): does each protocol learn the two tasks?
2. \(E_S^{W\text{-only}}\): is old-task degradation caused by \(w\), or did \(W\) drift?
3. \(\rho_W,\rho_w\): which teacher component is each parameter block recovering?
4. \(\Delta_{\rm eff}\): does LoRA learning correlate with the extensive-rank nuisance dropping?
5. \(C_{W\leftarrow w_*}\) and \(C_{w\leftarrow W_*}\): is there cross-contamination under joint or mismatched training?
6. \(\Gamma_W,\Gamma_w\): do the task gradients become conflicting around the same times that forgetting appears?

Only after the single-seed dynamics look sensible should the main conclusions be based on several seeds and the mean ± standard-deviation bands.
